In [1]:
%load_ext autoreload 
%autoreload 2
    

In [3]:
from elasticdino.training.ablations import train_depth, AblateDeformations, AblationTaskHeads, HypersimTaskHeads
from elasticdino.data.hypersim_depth import get_hypersim_datasets
from elasticdino.model.elasticdino import ElasticDino
import torch
import os

hypersim_path = "/mnt/home/mizrahiulysse/datasets/hypersim-depth"
   
model_config = dict(
    dino_model="b",
    n_features_in=768,
    layers={
        32: dict(hidden_features=256, n_blocks=4, layers_per_block=4),
        64: dict(hidden_features=256, n_blocks=4, layers_per_block=3),
        128: dict(hidden_features=128, n_blocks=4, layers_per_block=2),
    },
    start_size=32,
    target_size=128,
)

dino_repo = "/mnt/home/mizrahiulysse/model_cache/torch/hub/facebookresearch_dinov2_main"

def get_model(ablation):
    def get_ablated():
        if ablation == "heads":
            return ElasticDino(model_config, dino_repo)
        elif ablation == "deformations":
            return AblateDeformations(model_config, dino_repo)
        elif ablation == "none":
            return ElasticDino(model_config, dino_repo)
        else:
            raise Exception("Unknown ablation")
    
    return get_ablated


ablation = "deformations"

project_folder=f"/mnt/home/mizrahiulysse/elasticdino-runs/ablations-eval/{ablation}"

checkpoints = dict(
    heads=f"/mnt/home/mizrahiulysse/elasticdino-runs/ablations/heads/2025-01-29-06:15:35/checkpoints/60000/model.safetensors",
    deformations=f"/mnt/home/mizrahiulysse/elasticdino-runs/ablations/deformations/2025-01-29-05:45:20/60000/model.safetensors",
    none=f"/mnt/home/mizrahiulysse/elasticdino-runs/ablations/none/2025-01-29-05:44:35/checkpoints/60000/model.safetensors",
)

train_config = dict(
  n_epochs=50,
#   max_iterations=2,
  lr = 1e-3,
  debug_interval=100,
  n_features=64,
  save_interval=1000,
  batch_size=32,
  use_blur_loss=False,
  project_folder=project_folder,
  checkpoint=checkpoints[ablation]
)

def get_dataloaders():
    train, val = get_hypersim_datasets(hypersim_path)
    return torch.utils.data.DataLoader(train, batch_size=train_config["batch_size"], shuffle=False, num_workers=32), torch.utils.data.DataLoader(val, batch_size=train_config["batch_size"], shuffle=False, num_workers=32), 


args = (train_config, model_config, get_model(ablation), get_dataloaders)

from accelerate import notebook_launcher

notebook_launcher(
  train_depth,
  args,
  num_processes=2
)


Launching training on 2 GPUs.
b /mnt/home/mizrahiulysse/model_cache/torch/hub/facebookresearch_dinov2_main


W0218 00:09:56.299000 1506943 site-packages/torch/multiprocessing/spawn.py:160] Terminating process 1509727 via signal SIGTERM
E0218 00:09:56.611000 1506943 site-packages/torch/distributed/elastic/multiprocessing/api.py:732] failed (exitcode: 1) local_rank: 0 (pid: 1509726) of fn: train_depth (start_method: fork)
E0218 00:09:56.611000 1506943 site-packages/torch/distributed/elastic/multiprocessing/api.py:732] Traceback (most recent call last):
E0218 00:09:56.611000 1506943 site-packages/torch/distributed/elastic/multiprocessing/api.py:732]   File "/mnt/projects/conda-envs/dgxenv-2025-02-14-14-28-13-x8619-centos9-py310-pt251cf/lib/python3.10/site-packages/torch/distributed/elastic/multiprocessing/api.py", line 687, in _poll
E0218 00:09:56.611000 1506943 site-packages/torch/distributed/elastic/multiprocessing/api.py:732]     self._pc.join(-1)
E0218 00:09:56.611000 1506943 site-packages/torch/distributed/elastic/multiprocessing/api.py:732]   File "/mnt/projects/conda-envs/dgxenv-2025-02-1

ChildFailedError: 
============================================================
train_depth FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2025-02-18_00:09:56
  host      : pit103-slurm3009.thefacebook.com
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 1509726)
  error_file: /tmp/torchelastic_q5esot3l/none_3c8515_b/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/mnt/projects/conda-envs/dgxenv-2025-02-14-14-28-13-x8619-centos9-py310-pt251cf/lib/python3.10/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 355, in wrapper
      return f(*args, **kwargs)
    File "/mnt/home/mizrahiulysse/ElasticDino/elasticdino/training/ablations.py", line 346, in train_depth
      upscaler = get_model()
    File "/tmp/ipykernel_1506943/4216275115.py", line 28, in ge